# Geopack SDK: Advanced Processing with Workflows

This notebook covers how to use the **GDAL/OGR Workflow Framework** via the SDK to perform server-side processing like generating Hillshades, Contours, or Clipping datasets.

---

### 🚀 Setup Note
This notebook is configured to work both with an installed `geopack-sdk` package or directly from the repository source code (`src/` folder).

---

In [ ]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from dotenv import load_dotenv

# --- SMART SOURCE IMPORT ---
try:
    current_dir = os.getcwd()
    source_path = os.path.abspath(os.path.join(current_dir, "..", "src"))
    if os.path.exists(source_path):
        if source_path not in sys.path:
            sys.path.insert(0, source_path)
        print(f"ℹ️ Using SDK from local source: {source_path}")
    else:
        print("ℹ️ Using SDK from installed site-packages (pip)")
except Exception:
    print("⚠️ Could not determine local source path, falling back to pip.")

from geopack_sdk import (
    GeopackClient,
    GeopackAPIError,
    GeopackAuthError,
    GeopackError,
    GeopackTaskError,
    GeopackTimeoutError,
    task_log_entries_needing_review,
    task_message_badge_severity,
    task_message_info,
)

load_dotenv()
client = GeopackClient(base_url=os.getenv("GEOPACK_API_URL", "http://localhost:3000/api"))

# Login using credentials from .env or defaults
try:
    client.auth.login(
        os.getenv("GEOPACK_USERNAME", "admin"), 
        os.getenv("GEOPACK_PASSWORD", "password")
    )
    print("✅ Login successful!")
except GeopackAuthError as e:
    print(f"❌ Authentication failed (HTTP {e.status_code}): {e.message}")
except GeopackAPIError as e:
    print(f"❌ API error during login (HTTP {e.status_code}): {e.message}")
except GeopackError as e:
    print(f"❌ Login failed: {e.message}")


ℹ️ Using SDK from local source: d:\Works\geopack-geoportal\geopack-geoportal-v2\python-sdk\src
✅ Login successful!


## Handling SDK Errors

The SDK raises typed exceptions instead of generic `Exception`. Use them to branch on auth, API, task, and timeout failures.


In [ ]:
def report_sdk_error(context: str, exc: Exception) -> None:
    """Print a readable message for Geopack SDK exceptions."""
    if isinstance(exc, GeopackAuthError):
        print(f"[{context}] Auth error (HTTP {exc.status_code}): {exc.message}")
    elif isinstance(exc, GeopackAPIError):
        print(f"[{context}] API error (HTTP {exc.status_code}): {exc.message}")
    elif isinstance(exc, GeopackTaskError):
        print(f"[{context}] Task {exc.task_id} {exc.status}: {exc.message}")
    elif isinstance(exc, GeopackTimeoutError):
        print(f"[{context}] Timeout: {exc.message}")
    elif isinstance(exc, GeopackError):
        print(f"[{context}] {exc.message}")
    else:
        print(f"[{context}] Unexpected: {exc}")


# Example: not-found dataset -> GeopackAPIError (often 404)
try:
    client.datasets.get(999999999)
except GeopackAPIError as e:
    report_sdk_error("datasets.get", e)
    print(f"  status_code={e.status_code}")



## 1. Discovering Workflows
List the processing models available in your portal.

In [ ]:
# 1. Fetch available processing workflows
workflows = client.workflows.list()

if not workflows:
    print("❌ No workflows found in this portal.")
else:
    print(f"Available Workflows ({len(workflows)}):")
    for wf in workflows:
        print(f"- [{wf.id}] {wf.name}")

    # --- SELECT YOUR WORKFLOW HERE ---
    # Default to the first one, or change this to the desired ID (e.g. Hillshade, Contours)
    SELECTED_WF_ID = workflows[0].id
    print(f"\n✅ Selected Workflow ID: {SELECTED_WF_ID}")


Available Workflows (10):
- [31] Hillshade
- [42] مالتی بند
- [17] wkt reproject
- [41] xy-dem
- [14] dissolve
- [40] 🔥 Heatmap 
- [23] Multi criteria overlay
- [39] مکانیابی محل پارکینگ دوچرخه
- [24] Contour to DEM
- [38] Raster to Polygon

✅ Selected Workflow ID: 31


## 2. Inspecting a Workflow (e.g., Hillshade)
Let's check the required parameters for a processing model.

In [4]:
# Fetch details for the selected workflow
wf_details = client.workflows.get(SELECTED_WF_ID)

print(f"Workflow: {wf_details.name}")
params = client.workflows.extract_params(wf_details)

print("\nRequired/Optional Parameters:")
if not params:
    print("- (No parameters required)")
else:
    for p in params:
        print(f"- {p.key} ({p.type}): Default={p.default or 'None'}")


Workflow: Hillshade

Required/Optional Parameters:
- alt (number): Default=20
- azimuth (number): Default=320


In [ ]:
from typing import Optional


def inspect_workflow_run_outcome(client, task_id: str, run_id: Optional[int] = None) -> None:
    """
    Two layers (same as the portal):
    1) Background task `workflow:run` — messages, task.status, task.results
    2) WorkflowRun record — run.status, nodeStatuses, artifacts
    """
    task = client.tasks.get_status(task_id)
    info = task_message_info(task)
    severity = task_message_badge_severity(task)

    print(f"Task {task_id} ({task.taskType}): status={task.status}, badge={severity}, messages={info.count}")

    if severity in ("error", "warn"):
        print("  Task log has warn/error lines (may appear even when run status is succeeded):")
        for line in task_log_entries_needing_review(task):
            print(f"    [{line.get('level')}] {line.get('timestamp')} {line.get('message')}")

    if task.results:
        print(f"  task.results: {task.results}")

    wr_id = run_id
    if wr_id is None and task.inputParameters:
        wr_id = task.inputParameters.get("workflowRunId")
    if wr_id is None and isinstance(task.results, dict):
        wr_id = task.results.get("workflowRunId")

    if wr_id is None:
        print("  (No workflowRunId — pass run_id from submit response)")
        return

    run = client.workflow_runs.get(wr_id)
    logs = client.workflow_runs.get_logs(wr_id)
    print(f"\nWorkflowRun #{wr_id}: status={run.status}")

    if logs.get("error"):
        print(f"  run error: {logs['error']}")
    node_statuses = logs.get("nodeStatuses") or {}
    failed_nodes = {k: v for k, v in node_statuses.items() if str(v).lower() in ("failed", "error")}
    if failed_nodes:
        print(f"  failed nodes: {failed_nodes}")

    if run.artifacts:
        print(f"  artifacts: {len(run.artifacts)}")

## 3. Executing the Workflow

Each run is backed by a background task (`taskType`: **`workflow:run`**). The API returns **`workflowRunId`** and **`taskId`**.

- **Task** (`GET /tasks/{taskId}`): message log (warn/error like Task History), `task.results`
- **Workflow run** (`GET /workflow-runs/{id}`): run status, artifacts, node-level errors via `get_logs()`

A run can show **`succeeded`** while the task log still has warn/error lines — always inspect both (see `inspect_workflow_run_outcome` below).

In [5]:
print(f"Starting execution for Workflow #{SELECTED_WF_ID}...")

# Step 1: submit — keep taskId + workflowRunId (wait=False returns the 202 payload)
kickoff = client.workflow_runs.submit(
    workflow_id=SELECTED_WF_ID,
    params={},
    wait=False,
)
task_id = kickoff.taskId
run_id = getattr(kickoff, "workflowRunId", None)
print(f"  workflowRunId={run_id}, taskId={task_id}")

# Step 2: wait on the background task (same BullMQ job as the portal)
try:
    client.tasks.wait_for_task(task_id, quiet=False)
except GeopackTaskError as e:
    report_sdk_error("workflow:run", e)
    inspect_workflow_run_outcome(client, task_id, run_id)
    raise

# Step 3: inspect task log + workflow run (catches warn/error hidden behind succeeded)
inspect_workflow_run_outcome(client, task_id, run_id)

run_result = client.workflow_runs.get(run_id)
print(f"\n✅ WorkflowRun #{run_id} status: {run_result.status}")


Starting execution for Workflow #31...

✅ Execution finished with status: succeeded


### 3.1 Re-check a past workflow run by task id

If you already have a `taskId` from Task History (e.g. a run that created or updated a workflow), fetch the task first — `inputParameters` usually contains `workflowId` and `workflowRunId`.

In [ ]:
# List recent workflow:run tasks, or set PAST_TASK_ID from the portal Task History
PAST_TASK_ID = None  # e.g. "e163e3fe-1160-4816-9275-1c0be1be85b2"

if PAST_TASK_ID is None:
    recent = client.tasks.list(page_size=5, task_type="workflow:run")
    if recent.tasks:
        PAST_TASK_ID = recent.tasks[0].taskId
        print(f"Using latest workflow:run task: {PAST_TASK_ID}")
    else:
        print("No workflow:run tasks found for this user.")

if PAST_TASK_ID:
    inspect_workflow_run_outcome(client, PAST_TASK_ID)

## 4. Retrieving Artifacts
Processing workflows produce files (artifacts). Let's list and download them.

In [6]:
import os
artifacts = run_result.artifacts
file_artifacts = [a for a in artifacts if a.filePath]

if file_artifacts:
    target = file_artifacts[0]
    print(f"Found file artifact: {target.filePath}")
    
    # Download to local directory
    os.makedirs("output", exist_ok=True)
    local_path = client.workflow_runs.download_artifact(run_result.id, target.id, "output/")
    print(f"✅ Downloaded to: {local_path}")
else:
    print("No file artifacts produced (maybe it was a Dataset registration only).")


Found file artifact: workflow-runs\126\hillshade_1_0_hillshade_1_output.tif
✅ Downloaded to: d:\Works\geopack-geoportal\geopack-geoportal-v2\python-sdk\notebooks\output\hillshade_1_0_hillshade_1_output.tif
